This notebook appends the `ICH1` and `ICH2` Q75 forecasts from `station2170_q_quantiles_by_model.csv` into `station2170_deterministic.csv`, renames them to `ICH1_Q75` and `ICH2_Q75`, and fills `H` with `NaN` for the new rows.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_project_root(start=None):
    """Find the analysis workspace root by looking for the outputs folder."""
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "outputs").exists() and (candidate / "outputs" / "OFEV_probabilistic").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root containing outputs/OFEV_probabilistic.")


root = find_project_root()
source_dir = root / "outputs" / "OFEV_probabilistic"
det_path = source_dir / "station2170_deterministic.csv"
q75_path = source_dir / "station2170_q_quantiles_by_model.csv"
out_path = source_dir / "station2170_deterministic_with_q75.csv"

# Load source tables

det = pd.read_csv(det_path)
q75 = pd.read_csv(q75_path)

det.columns = det.columns.str.strip()
q75.columns = q75.columns.str.strip()

# Parse datetimes for safe matching
for frame in (det, q75):
    frame["issue_time"] = pd.to_datetime(frame["issue_time"])
    frame["valid_time"] = pd.to_datetime(frame["valid_time"])
    if "lead_time_h" in frame.columns:
        frame["lead_time_h"] = pd.to_numeric(frame["lead_time_h"], errors="coerce")

# Keep only the Q75 rows for ICH1 and ICH2
q75_insert = q75[q75["model"].isin(["ICH1", "ICH2"])].copy()
q75_insert = q75_insert.rename(columns={"Q_p75": "Q"})
q75_insert["model"] = q75_insert["model"] + "_Q75"
q75_insert["H"] = np.nan
q75_insert["source_path"] = pd.NA

# Reorder to match the deterministic file layout
q75_insert = q75_insert[det.columns]

# Append the new rows to the deterministic table
merged = pd.concat([det, q75_insert], ignore_index=True)
merged = merged.sort_values(["model", "issue_time", "valid_time", "lead_time_h"], kind="stable")
merged = merged.reset_index(drop=True)

# Save the new file
merged.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Deterministic rows: {len(det):,}")
print(f"Inserted Q75 rows: {len(q75_insert):,}")
print(f"Final rows: {len(merged):,}")
print("New models:", sorted(merged.loc[merged["model"].str.endswith("_Q75"), "model"].unique()))

Saved: C:\Users\simon\Documents\DP\analysis\outputs\OFEV_probabilistic\station2170_deterministic_with_q75.csv
Deterministic rows: 64,803
Inserted Q75 rows: 4,656
Final rows: 69,459
New models: ['ICH1_Q75', 'ICH2_Q75']


In [ ]:
# Quick verification of the inserted rows
check = merged[merged["model"].isin(["ICH1_Q75", "ICH2_Q75"])].copy()

print(check[["model", "issue_time", "valid_time", "lead_time_h", "H", "Q"]].head(20).to_string(index=False))
print()
print("Rows with missing H in inserted Q75 rows:", check["H"].isna().sum())
print("Unique inserted models:", check["model"].unique().tolist())
print("Matching issue_time/valid_time/lead_time combinations:", check[["issue_time", "valid_time", "lead_time_h"]].drop_duplicates().shape[0])